# ContinualRAG-BEIR (v12 Official Paper Release): Bounded-Memory Retrieval Under Sequential Domain Shift

**Central Question.** When a Retrieval-Augmented Generation (RAG) agent operates over a sequential stream of knowledge domains under a strict document memory budget ($M_{\max} = 40\%$), which documents should it keep and which should it evict to balance **skewed repeated traffic (exploitation)** against **unseen novel queries (exploration)**?

**Benchmark Streams.**
* **Base-5 Stream:** 5 sequentially arriving BEIR domains (`scifact`, `nfcorpus`, `fiqa`, `arguana`, `scidocs`; $5,000$ streaming documents, memory cap $M_{\max} = 2,000$).
* **Extended-8 Stream:** 8 sequentially arriving BEIR domains (`scifact`, `nfcorpus`, `fiqa`, `arguana`, `scidocs`, `trec-covid`, `quora`, `webis-touche2020`; $8,000$ streaming documents, memory cap $M_{\max} = 3,200$).

**Leak-Free Protocol & Pre-Registered Governance.**
* Per domain, queries are partitioned into a **Usage** pool ($50\%$, sampled via a Zipfian stream $p(q) \propto \text{rank}^{-1.0}$ for online feedback) and a strictly disjoint **Test** pool ($50\%$, never observed by any policy).
* **Cross-Embedder Validity Control:** Policies select documents using `intfloat/e5-small-v2` (`SEL_EMBED_MODEL`) while retrieval and evaluation use `BAAI/bge-small-en-v1.5` (`EMBED_MODEL`), eliminating self-selection circularity and BEIR pooling bias.
* **Statistical Governance:** Seeds `0-9` are *exploratory*. Seeds `100-719` (85 distinct seeds across 7 confirmatory blocks) are *locked confirmatory* cohorts evaluated with Holm-Bonferroni corrected two-sided Wilcoxon signed-rank tests and 5,000-sample bootstrap 95% confidence intervals.


In [ ]:
!pip -q install sentence-transformers scipy pandas seaborn matplotlib tqdm

In [ ]:
import os, io, json, time, math, zipfile, hashlib, platform, urllib.request, warnings
from dataclasses import dataclass, field, replace, asdict
from typing import Dict, List, Tuple, Optional
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.linalg import solve_triangular
from scipy.stats import wilcoxon, rankdata

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

# ==============================================================================
# EXPERIMENT CONFIGURATION
# ==============================================================================
MODE = "smoke"                           # "smoke" (synthetic fast check, ~1 min) | "full" (real BEIR data; GPU recommended)
EMBED_MODEL = "BAAI/bge-small-en-v1.5"   # Primary retrieval & evaluation embedder
SEL_EMBED_MODEL = "intfloat/e5-small-v2" # Cross-embedder selection control (set to None for same-embedder BGE->BGE regime)

DOMAIN_SETS = {
    "base5": ["scifact", "nfcorpus", "fiqa", "arguana", "scidocs"],
    "extended8": ["scifact", "nfcorpus", "fiqa", "arguana", "scidocs", "trec-covid", "quora", "webis-touche2020"],
}
DOMAIN_SET = "base5"
DOMAIN_POOL_CAP = {"trec-covid": 10}

EMBED_PRESETS = {
    "BAAI/bge-small-en-v1.5": ("Represent this sentence for searching relevant passages: ", ""),
    "sentence-transformers/all-MiniLM-L6-v2": ("", ""),
    "intfloat/e5-small-v2": ("query: ", "passage: "),
}

@dataclass
class Config:
    domains: List[str] = field(default_factory=lambda: ["scifact", "nfcorpus", "fiqa", "arguana", "scidocs"])
    embed_model: str = "BAAI/bge-small-en-v1.5"
    query_prefix: str = "Represent this sentence for searching relevant passages: "
    doc_prefix: str = ""
    sel_model: Optional[str] = None
    cache_budget: Optional[int] = None
    max_seq_len: int = 256
    pool_neg: int = 1500
    docs_per_domain: int = 1000
    max_queries_per_domain: int = 200
    max_rel_per_query: int = 5
    rel_budget_frac: float = 0.6
    usage_frac: float = 0.5
    n_usage_per_phase: int = 150
    zipf_s: float = 1.0
    top_k: int = 10
    cap: int = 2000
    decay: float = 0.8
    qprox_tau: float = 0.05
    mix_rhos: List[float] = field(default_factory=lambda: [0.1, 0.25, 0.5])
    seeds: List[int] = field(default_factory=lambda: list(range(10)))
    sweep_seeds: List[int] = field(default_factory=lambda: list(range(5)))
    feedback_modes: List[str] = field(default_factory=lambda: ["relevance", "retrieval"])
    out_dir: str = "results_v12"
    cache_dir: str = "cache_v6"
    data_dir: str = "beir_data"

_qp, _dp = EMBED_PRESETS[EMBED_MODEL]
assert SEL_EMBED_MODEL is None or SEL_EMBED_MODEL in EMBED_PRESETS, "Add model and prefixes to EMBED_PRESETS"
_doms = DOMAIN_SETS[DOMAIN_SET]

CFG = Config(
    domains=list(_doms),
    embed_model=EMBED_MODEL,
    query_prefix=_qp,
    doc_prefix=_dp,
    sel_model=SEL_EMBED_MODEL,
    cap=int(0.4 * 1000 * len(_doms)),
    out_dir="results_v12_" + EMBED_MODEL.split("/")[-1]
            + (("_sel-" + SEL_EMBED_MODEL.split("/")[-1]) if SEL_EMBED_MODEL else "")
            + ("" if DOMAIN_SET == "base5" else "_" + DOMAIN_SET),
)

if MODE == "smoke":
    CFG = replace(
        CFG,
        docs_per_domain=300,
        max_queries_per_domain=60,
        cap=600,
        n_usage_per_phase=60,
        seeds=list(range(4)),
        sweep_seeds=[0, 1],
        out_dir="results_smoke_v12",
    )

os.makedirs(CFG.out_dir, exist_ok=True)
os.makedirs(CFG.cache_dir, exist_ok=True)
print("Active Mode:", MODE, "\nConfig:", asdict(CFG))

## 1. Related Literature & Mechanistic Motivation
* **Continual Retrieval & Graph Memory:** *HippoRAG 2* (Gutiérrez et al., ICML 2025) models non-parametric continual memory via Personalized PageRank (`GraphPPR+Quota`). *RAG without Forgetting* (Hu et al., arXiv:2602.05152, 2026) evolves document keys toward answered queries. *CREAM* (Son et al., arXiv:2601.02708, 2026) trains streaming encoders using stratified coreset sampling (`K-Center+Quota`). *Streaming RAG* (Zhu, arXiv:2508.05662, 2025) applies heavy-hitter filtering under a memory budget.
* **Frequency Caching:** *TinyLFU / W-TinyLFU* (Einziger et al., ACM ToS 2017) combines recency admission windows with frequency sketches (`WindowHit`).
* **Hubness & Static Pruning:** Radovanović et al. (JMLR 2010) proved that high-dimensional cosine spaces produce *hubs* (points appearing in many $k$-NN lists) and *anti-hubs* (outliers that almost never appear in any top-$k$ list). Static index pruning (Carmel et al., SIGIR 2001; Chang et al., SIGIR 2024) removes low-utility documents prior to deployment.
* **BEIR Pooling Bias Control:** Thakur et al. (NeurIPS 2021) noted that BEIR judgments come from pooling existing retrievers. Selecting documents with `intfloat/e5-small-v2` (`SEL_EMBED_MODEL`) while retrieving with `BAAI/bge-small-en-v1.5` (`EMBED_MODEL`) eliminates retriever self-selection circularity.

In [ ]:
@dataclass
class Pool:
    name: str
    D: np.ndarray                              # (n_docs, dim) L2-normalised retrieval doc embeddings
    Q: np.ndarray                              # (n_q, dim)    L2-normalised retrieval query embeddings
    qrels: Dict[int, List[Tuple[int, float]]]  # query -> [(doc, grade)], grade > 0, sorted desc
    selfdoc: Dict[int, int]                    # query -> doc with identical id (ArguAna)
    n_docs: int = 0
    Ds: Optional[np.ndarray] = None            # selection-model doc embeddings (None = same as D)
    Qs: Optional[np.ndarray] = None

BEIR_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip"

def download_beir(name: str) -> str:
    path = os.path.join(CFG.data_dir, name)
    if os.path.exists(os.path.join(path, "corpus.jsonl")):
        return path
    os.makedirs(CFG.data_dir, exist_ok=True)
    zp = os.path.join(CFG.data_dir, name + ".zip")
    print(f"Downloading {name} ...")
    urllib.request.urlretrieve(BEIR_URL.format(name), zp)
    with zipfile.ZipFile(zp) as z:
        z.extractall(CFG.data_dir)
    os.remove(zp)
    return path

def load_beir(name: str):
    p = download_beir(name)
    corpus = {}
    with open(os.path.join(p, "corpus.jsonl"), encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            corpus[r["_id"]] = ((r.get("title") or "") + ". " + (r.get("text") or "")).strip()
    queries = {}
    with open(os.path.join(p, "queries.jsonl"), encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            queries[r["_id"]] = r["text"]
    qrels = {}
    with open(os.path.join(p, "qrels", "test.tsv"), encoding="utf-8") as f:
        next(f)
        for line in f:
            q, d, s = line.rstrip("\n").split("\t")[:3]
            qrels.setdefault(q, {})[d] = float(s)
    return corpus, queries, qrels

def r_ids(r):
    return [d for d, _ in r]

def _cache_path(name: str, model: str) -> str:
    qp, dp = EMBED_PRESETS[model]
    cap = DOMAIN_POOL_CAP.get(name)
    key = f"{name}|{model}|{CFG.pool_neg}|{CFG.max_seq_len}|{qp}|{dp}" + (f"|cap{cap}" if cap else "")
    return os.path.join(CFG.cache_dir, f"{name}_{hashlib.md5(key.encode()).hexdigest()[:10]}.npz")

def _embed_cached(name: str, model: str, fn, doc_texts: List[str], query_texts: List[str]):
    cpath = _cache_path(name, model)
    if os.path.exists(cpath):
        z = np.load(cpath)
        return z["D"], z["Q"]
    print(f"Embedding {name} with {model}: {len(doc_texts)} docs, {len(query_texts)} queries")
    D = fn(doc_texts, is_query=False)
    Q = fn(query_texts, is_query=True)
    np.savez_compressed(cpath, D=D, Q=Q)
    return D, Q

def build_pool_real(name: str, emb_ret, emb_sel=None) -> Pool:
    corpus, queries, qrels = load_beir(name)
    cap_pp = DOMAIN_POOL_CAP.get(name)
    prng = np.random.RandomState(1)
    pos = {}
    for q, rel in qrels.items():
        if q in queries:
            r = [(d, s) for d, s in rel.items() if s > 0 and d in corpus]
            if not r:
                continue
            if cap_pp and len(r) > cap_pp:
                r = [r[i] for i in prng.permutation(len(r))]
            r = sorted(r, key=lambda x: -x[1])
            pos[q] = r[:cap_pp] if cap_pp else r
    pos_docs = sorted({d for r in pos.values() for d in r_ids(r)})
    rng = np.random.RandomState(0)
    pos_set = set(pos_docs)
    others = sorted(d for d in corpus if d not in pos_set)
    neg = list(rng.choice(others, size=min(CFG.pool_neg, len(others)), replace=False)) if others else []
    doc_ids = pos_docs + neg
    id2loc = {d: i for i, d in enumerate(doc_ids)}
    qids = sorted(pos)
    dt, qt = [corpus[d] for d in doc_ids], [queries[q] for q in qids]
    D, Q = _embed_cached(name, CFG.embed_model, emb_ret, dt, qt)
    Ds = Qs = None
    if emb_sel is not None and CFG.sel_model and CFG.sel_model != CFG.embed_model:
        Ds, Qs = _embed_cached(name, CFG.sel_model, emb_sel, dt, qt)
        Ds, Qs = Ds.astype(np.float32), Qs.astype(np.float32)
    qrels_loc = {i: [(id2loc[d], s) for d, s in pos[q]] for i, q in enumerate(qids)}
    selfdoc = {i: id2loc[q] for i, q in enumerate(qids) if q in id2loc}
    st = np.mean([len(v) for v in qrels_loc.values()])
    reuse = np.mean(np.bincount([d for v in qrels_loc.values() for d, _ in v]) > 1)
    print(f"  {name}: pool docs={len(doc_ids)} queries={len(qids)} avg positives/query={st:.1f} pos shared by >1 query={reuse:.2f}")
    return Pool(name, D.astype(np.float32), Q.astype(np.float32), qrels_loc, selfdoc, len(doc_ids), Ds, Qs)

def make_embedder(model_name: str):
    from sentence_transformers import SentenceTransformer
    import torch
    qp, dp = EMBED_PRESETS[model_name]
    m = SentenceTransformer(model_name, device="cuda" if torch.cuda.is_available() else "cpu")
    m.max_seq_length = CFG.max_seq_len
    def f(texts, is_query):
        texts = [(qp if is_query else dp) + t for t in texts]
        return m.encode(texts, batch_size=128, normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True)
    return f

def build_pool_smoke(name: str, seed: int, dim: int = 48) -> Pool:
    """Synthetic BEIR-like pool used only in smoke mode to verify the pipeline."""
    rng = np.random.RandomState(int(hashlib.md5(name.encode()).hexdigest(), 16) % (2**31) + seed)
    centre = rng.randn(dim); centre /= np.linalg.norm(centre)
    n_topics, per_topic = 100, 3
    topics = centre * 1.2 + rng.randn(n_topics, dim) * 0.5
    docs, qrels, qs = [], {}, []
    for t in range(n_topics):
        base = len(docs)
        for _ in range(per_topic):
            docs.append(topics[t] + rng.randn(dim) * 0.35)
        for _ in range(2):
            qs.append(topics[t] + rng.randn(dim) * 0.35)
            qrels[len(qs) - 1] = [(base + k, 1.0) for k in range(per_topic)]
    for _ in range(300):
        docs.append(centre * 1.2 + rng.randn(dim) * 0.9)
    D = np.array(docs, np.float32); Q = np.array(qs, np.float32)
    D /= np.linalg.norm(D, axis=1, keepdims=True)
    Q /= np.linalg.norm(Q, axis=1, keepdims=True)
    selfdoc = {0: 0} if name.startswith("D4") else {}
    Ds = Qs = None
    if SEL_EMBED_MODEL:
        rr = np.random.RandomState(7); R, _ = np.linalg.qr(rr.randn(dim, dim))
        Ds = D @ R + 0.15 * rr.randn(*D.shape).astype(np.float32)
        Qs = Q @ R + 0.15 * rr.randn(*Q.shape).astype(np.float32)
        Ds = (Ds / np.linalg.norm(Ds, axis=1, keepdims=True)).astype(np.float32)
        Qs = (Qs / np.linalg.norm(Qs, axis=1, keepdims=True)).astype(np.float32)
    return Pool(name, D, Q, qrels, selfdoc, len(docs), Ds, Qs)

if MODE == "full":
    EMB = make_embedder(CFG.embed_model)
    EMB_SEL = make_embedder(CFG.sel_model) if (CFG.sel_model and CFG.sel_model != CFG.embed_model) else None
    POOLS = [build_pool_real(d, EMB, EMB_SEL) for d in CFG.domains]
else:
    POOLS = [build_pool_smoke(f"D{i}", 0) for i in range(len(CFG.domains))]

DOC_OFF = np.cumsum([0] + [p.n_docs for p in POOLS])
Q_OFF = np.cumsum([0] + [p.Q.shape[0] for p in POOLS])
DEMB = np.vstack([p.D for p in POOLS])
QEMB = np.vstack([p.Q for p in POOLS])

if all(p.Ds is not None for p in POOLS):
    DSEL = np.vstack([p.Ds for p in POOLS])
    QSEL = np.vstack([p.Qs for p in POOLS])
else:
    DSEL, QSEL = DEMB, QEMB

N_DOCS = DEMB.shape[0]
print("Selection embeddings are", "a DIFFERENT model" if DSEL is not DEMB else "the same as retrieval embeddings")
print(f"Global docs: {N_DOCS} | Global queries: {QEMB.shape[0]} | Embedding dim: {DEMB.shape[1]}")

## 2. Episode Generator (One Seed): Task Order, Subsample, Disjoint Query Split, Zipf Usage Stream

In [ ]:
class Episode:
    def __init__(self, pools: List[Pool], cfg: Config, seed: int):
        self.cfg, self.seed = cfg, seed
        rng = np.random.RandomState(seed)
        order = rng.permutation(len(pools))
        self.T = len(pools)
        self.N = N_DOCS
        self.doc_dom = -np.ones(N_DOCS, int)
        self.doc_arr = -np.ones(N_DOCS, int)
        self.oracle_cnt = np.zeros(N_DOCS)
        self.tasks, self.names = [], []
        seq = 0
        for pos, pi in enumerate(order):
            P = pools[pi]
            do, qo = DOC_OFF[pi], Q_OFF[pi]
            qids = list(rng.permutation(len(P.qrels)))
            budget = int(cfg.rel_budget_frac * cfg.docs_per_domain)
            chosen, rel_union = [], set()
            for q in qids:
                r = list(P.qrels[q])
                rng.shuffle(r)
                r.sort(key=lambda x: -x[1])
                new = {d for d, _ in r[:cfg.max_rel_per_query]}
                if len(rel_union | new) > budget:
                    continue
                chosen.append((q, new))
                rel_union |= new
                if len(chosen) >= cfg.max_queries_per_domain:
                    break
            assert len(chosen) >= 20, f"Too few queries for {P.name}: {len(chosen)}"
            rest = np.setdiff1d(np.arange(P.n_docs), np.fromiter(rel_union, int))
            n_neg = max(0, cfg.docs_per_domain - len(rel_union))
            negs = rng.choice(rest, size=min(n_neg, len(rest)), replace=False)
            local_docs = rng.permutation(np.concatenate([np.fromiter(rel_union, int), negs]))
            docs = (local_docs + do).astype(int)
            self.doc_dom[docs] = pos
            self.doc_arr[docs] = np.arange(seq, seq + len(docs))
            seq += len(docs)
            recs = []
            for q, rel in chosen:
                relg = np.array(sorted(d + do for d in rel), int)
                sd = P.selfdoc.get(q, -1)
                recs.append(dict(q=int(q + qo), rel=relg, selfdoc=int(sd + do) if sd >= 0 else -1, dom=pos))
                self.oracle_cnt[relg] += 1
            n_use = max(10, int(cfg.usage_frac * len(recs)))
            self.tasks.append(dict(docs=docs, usage=recs[:n_use], test=recs[n_use:]))
            self.names.append(P.name)

        # Zipfian online usage stream (identical across all policies for a given seed)
        srng = np.random.RandomState(seed * 7919 + 13)
        self.pop = []
        for t in range(self.T):
            n = len(self.tasks[t]["usage"])
            w = 1.0 / (srng.permutation(n) + 1.0) ** cfg.zipf_s
            self.pop.append(w / w.sum())
        self.stream = [
            [srng.choice(len(self.tasks[j]["usage"]), size=cfg.n_usage_per_phase, p=self.pop[j]) for j in range(t + 1)]
            for t in range(self.T)
        ]

def retrieve(mem: np.ndarray, recs: List[dict], k: int) -> np.ndarray:
    """Exact cosine top-k over memory, excluding query self-documents (ArguAna)."""
    if len(recs) == 0 or len(mem) == 0:
        return np.zeros((len(recs), 0), int)
    qi = np.array([r["q"] for r in recs])
    S = QEMB[qi] @ DEMB[mem].T
    lut = -np.ones(N_DOCS, int)
    lut[mem] = np.arange(len(mem))
    sd = np.array([r["selfdoc"] for r in recs])
    ok = sd >= 0
    if ok.any():
        pos = lut[sd[ok]]
        rows = np.where(ok)[0][pos >= 0]
        S[rows, pos[pos >= 0]] = -np.inf
    ke = min(k, len(mem))
    part = np.argpartition(-S, ke - 1, axis=1)[:, :ke]
    order = np.argsort(-np.take_along_axis(S, part, 1), axis=1)
    return mem[np.take_along_axis(part, order, 1)]

def score_queries(ret: np.ndarray, recs: List[dict], k: int):
    """Binary-relevance nDCG@k, Recall@k, Success@k normalized over the full domain subsample."""
    disc = 1.0 / np.log2(np.arange(k) + 2.0)
    nd, rc, su = [], [], []
    for i, r in enumerate(recs):
        rel = np.isin(ret[i], r["rel"])
        L = len(rel)
        dcg = (rel * disc[:L]).sum()
        idcg = disc[:min(k, len(r["rel"]))].sum()
        nd.append(dcg / idcg)
        rc.append(rel.sum() / len(r["rel"]))
        su.append(float(rel.any()))
    return np.array(nd), np.array(rc), np.array(su)

## 3. Bounded-Memory Selection & Eviction Policies
All bounded policies observe only arriving documents, domain/arrival indices, and online usage-stream feedback (`relevance` or `retrieval`). Held-out `test` queries are never observed.

In [ ]:
class Policy:
    name = "Policy"
    def __init__(self, ep: Episode, cap: int, fb: str, seed: int):
        self.ep, self.cap, self.fb = ep, cap, fb
        self.rng = np.random.RandomState(seed * 31 + 5)
        self.mem = np.zeros(0, int)
        self.n_seen = 0
        self.phase = 0
        self.hits = np.zeros(N_DOCS)
        self.cnt_total = np.zeros(N_DOCS)
        self.last_hit = -np.ones(N_DOCS)
        self.qc = np.zeros(ep.T)
        self.exposed = np.zeros(N_DOCS, bool)
        self.qw = np.zeros(QEMB.shape[0])
        self.evict_time = 0.0

    def arrive(self, t: int, new: np.ndarray):
        t0 = time.perf_counter()
        self.n_seen += len(new)
        cand = np.concatenate([self.mem, new])
        self.mem = cand if len(cand) <= self.cap else self.select(cand, t)
        assert len(self.mem) <= self.cap and len(np.unique(self.mem)) == len(self.mem)
        self.evict_time += time.perf_counter() - t0

    def stage(self, t: int, new: np.ndarray):
        t0 = time.perf_counter()
        self.n_seen += len(new)
        self.mem = np.concatenate([self.mem, new])
        self.evict_time += time.perf_counter() - t0

    def consolidate(self, t: int):
        t0 = time.perf_counter()
        if len(self.mem) > self.cap:
            self.mem = self.select(self.mem, t)
        assert len(self.mem) <= self.cap and len(np.unique(self.mem)) == len(self.mem)
        self.evict_time += time.perf_counter() - t0

    def feedback(self, t: int, hit_docs: np.ndarray, qcounts: np.ndarray, qidx: np.ndarray):
        self.hits *= self.ep.cfg.decay
        self.qc *= self.ep.cfg.decay
        self.qw *= self.ep.cfg.decay
        np.add.at(self.qw, qidx, 1.0)
        if len(hit_docs):
            np.add.at(self.hits, hit_docs, 1.0)
            np.add.at(self.cnt_total, hit_docs, 1.0)
            self.last_hit[np.unique(hit_docs)] = self.phase
        self.qc += qcounts
        self.exposed[self.mem] = True
        self.phase += 1

    def rate(self, idx: np.ndarray) -> np.ndarray:
        return self.hits[idx] / np.maximum(self.qc[self.ep.doc_dom[idx]], 1.0)

    def recency_tiebreak(self, cand: np.ndarray) -> np.ndarray:
        return 1e-6 * self.ep.doc_arr[cand] / max(1, self.ep.doc_arr.max())

    def pick(self, cand: np.ndarray, scores: np.ndarray, quota: bool = False, n: Optional[int] = None) -> np.ndarray:
        n = self.cap if n is None else int(n)
        if n <= 0:
            return cand[:0]
        if n >= len(cand):
            return cand.copy()
        scores = scores + self.rng.rand(len(scores)) * 1e-9
        if not quota:
            return cand[np.argpartition(-scores, n - 1)[:n]]
        doms = self.ep.doc_dom[cand]
        ud = np.unique(doms)
        share = n // len(ud)
        chosen = []
        for d in ud:
            ii = np.where(doms == d)[0]
            chosen.append(ii[np.argsort(-scores[ii])][:share])
        chosen = np.concatenate(chosen)
        rest = np.setdiff1d(np.arange(len(cand)), chosen)
        need = n - len(chosen)
        if need > 0:
            chosen = np.concatenate([chosen, rest[np.argsort(-scores[rest])][:need]])
        return cand[chosen]

    def select(self, cand: np.ndarray, t: int) -> np.ndarray:
        raise NotImplementedError

# ---- Classic & Reference Policies ------------------------------------------------
class Unbounded(Policy):
    name = "Unbounded (reference)"
    def select(self, cand, t): return cand

class Oracle(Policy):
    name = "Oracle (uses test labels)"
    def select(self, cand, t): return self.pick(cand, self.ep.oracle_cnt[cand])

class FIFO(Policy):
    name = "FIFO"
    def select(self, cand, t): return self.pick(cand, self.ep.doc_arr[cand].astype(float))

class RandomEvict(Policy):
    name = "Random"
    def select(self, cand, t): return self.pick(cand, self.rng.rand(len(cand)))

class QuotaRandom(Policy):
    name = "Quota-Random"
    def select(self, cand, t): return self.pick(cand, self.rng.rand(len(cand)), quota=True)

class Reservoir(Policy):
    name = "Reservoir"
    def arrive(self, t, new):
        t0 = time.perf_counter(); mem = list(self.mem)
        for d in new:
            self.n_seen += 1
            if len(mem) < self.cap:
                mem.append(d)
            else:
                j = self.rng.randint(0, self.n_seen)
                if j < self.cap:
                    mem[j] = d
        self.mem = np.array(mem, int)
        self.evict_time += time.perf_counter() - t0

class LRU(Policy):
    name = "LRU"
    def select(self, cand, t): return self.pick(cand, self.last_hit[cand])

class LFU(Policy):
    name = "LFU"
    def select(self, cand, t): return self.pick(cand, self.cnt_total[cand])

class DecayedHit(Policy):
    name = "DecayedHit"
    quota = False; recent = False
    def select(self, cand, t):
        sc = self.rate(cand) + (self.recency_tiebreak(cand) if self.recent else 0.0)
        return self.pick(cand, sc, quota=self.quota)

class DecayedHitQuota(DecayedHit):
    name = "DecayedHit+Quota"; quota = True

class DecayedHitRecent(DecayedHit):
    name = "DecayedHit (recency tie-break)"; recent = True

class Mix(Policy):
    def __init__(self, ep, cap, fb, seed, rho=0.25):
        super().__init__(ep, cap, fb, seed); self.rho = rho
    def select(self, cand, t):
        rate = self.rate(cand); k = int(self.rho * self.cap)
        hot = np.where(rate > 0)[0]; hot = hot[np.argsort(-rate[hot])][:k]
        rest = np.setdiff1d(np.arange(len(cand)), hot)
        fill = self.rng.choice(rest, self.cap - len(hot), replace=False)
        return cand[np.concatenate([hot, fill])]

class WindowHit(Policy):
    name = "WindowHit (W-TinyLFU-style)"
    def __init__(self, ep, cap, fb, seed, w=0.25):
        super().__init__(ep, cap, fb, seed); self.w = w
    def select(self, cand, t):
        k = int(self.w * self.cap)
        newest = cand[np.argsort(-self.ep.doc_arr[cand])[:k]]
        rest = cand[~np.isin(cand, newest)]
        return np.concatenate([newest, self.pick(rest, self.rate(rest), n=self.cap - len(newest))])

# ---- Geometric, Coreset & Structural Priors --------------------------------------
def knn_density(E: np.ndarray, k: int = 5) -> np.ndarray:
    n = len(E); k = min(k, n - 1)
    if k < 1: return np.zeros(n)
    S = E @ E.T; np.fill_diagonal(S, -1.0)
    return np.partition(S, -k, axis=1)[:, -k:].mean(1)

def knn_indegree(E: np.ndarray, Q=None, w=None, k: int = 10) -> np.ndarray:
    n = len(E); k = min(k, n - 1)
    if k < 1: return np.zeros(n)
    S = E @ E.T; np.fill_diagonal(S, -1.0)
    nn = np.argpartition(-S, k, axis=1)[:, :k]
    return np.bincount(nn.ravel(), minlength=n).astype(float)

class Density(Policy):
    name = "Density"
    def __init__(self, ep, cap, fb, seed, quota=False):
        super().__init__(ep, cap, fb, seed); self.quota = quota
    def select(self, cand, t):
        if not self.quota:
            return self.pick(cand, knn_density(DSEL[cand]))
        sc = np.zeros(len(cand)); doms = self.ep.doc_dom[cand]
        for d in np.unique(doms):
            ii = np.where(doms == d)[0]
            sc[ii] = knn_density(DSEL[cand[ii]])
        return self.pick(cand, sc, quota=True)

class HitDensity(Policy):
    name = "HitDensity+Quota"
    def __init__(self, ep, cap, fb, seed, rho=0.25):
        super().__init__(ep, cap, fb, seed); self.rho = rho
    def select(self, cand, t):
        rate = self.rate(cand); k = int(self.rho * self.cap)
        hot = np.where(rate > 0)[0]; hot = hot[np.argsort(-rate[hot])][:k]
        rest = np.setdiff1d(np.arange(len(cand)), hot); R = self.cap - len(hot); rc = cand[rest]
        sc = np.zeros(len(rc)); doms = self.ep.doc_dom[rc]
        for d in np.unique(doms):
            ii = np.where(doms == d)[0]; sc[ii] = knn_density(DSEL[rc[ii]])
        return np.concatenate([cand[hot], self.pick(rc, sc, quota=True, n=R)])

def kcenter_idx(E: np.ndarray, n: int, rng) -> np.ndarray:
    N = len(E)
    if n >= N: return np.arange(N)
    c = int(rng.randint(N)); chosen = [c]
    maxsim = E @ E[c]; maxsim[c] = np.inf
    for _ in range(n - 1):
        c = int(np.argmin(maxsim)); chosen.append(c)
        maxsim = np.maximum(maxsim, E @ E[c]); maxsim[c] = np.inf
    return np.array(chosen)

class KCenter(Policy):
    name = "K-Center (diversity)"
    def __init__(self, ep, cap, fb, seed, quota=False):
        super().__init__(ep, cap, fb, seed); self.quota = quota
    def select(self, cand, t):
        E = DSEL[cand]
        if not self.quota:
            return cand[kcenter_idx(E, self.cap, self.rng)]
        doms = self.ep.doc_dom[cand]; ud = np.unique(doms); share = self.cap // len(ud); chosen = []
        for d in ud:
            ii = np.where(doms == d)[0]
            chosen.append(ii[kcenter_idx(E[ii], share, self.rng)])
        chosen = np.concatenate(chosen); need = self.cap - len(chosen)
        if need > 0:
            rest = np.setdiff1d(np.arange(len(cand)), chosen)
            chosen = np.concatenate([chosen, self.rng.choice(rest, need, replace=False)])
        return cand[chosen]

class QueryProx(Policy):
    name = "QueryProx"
    def __init__(self, ep, cap, fb, seed, quota=False):
        super().__init__(ep, cap, fb, seed); self.quota = quota
    def select(self, cand, t):
        lg = np.where(self.qw > 1e-3)[0]
        if len(lg) < 20:
            return self.pick(cand, self.rng.rand(len(cand)), self.quota)
        from scipy.special import logsumexp
        S = (DSEL[cand] @ QSEL[lg].T) / self.ep.cfg.qprox_tau
        return self.pick(cand, logsumexp(S + np.log(self.qw[lg])[None, :], axis=1), self.quota)

class ScorePolicy(Policy):
    def __init__(self, ep, cap, fb, seed, score_fn, per_domain=True, quota=True):
        super().__init__(ep, cap, fb, seed)
        self.score_fn, self.per_domain, self.quota = score_fn, per_domain, quota
    def select(self, cand, t):
        lg = np.where(self.qw > 1e-3)[0]; Q, w = QSEL[lg], self.qw[lg]
        if not self.per_domain:
            return self.pick(cand, np.asarray(self.score_fn(DSEL[cand], Q, w), float), self.quota)
        sc = np.zeros(len(cand)); doms = self.ep.doc_dom[cand]
        for d in np.unique(doms):
            ii = np.where(doms == d)[0]
            sc[ii] = rankdata(np.asarray(self.score_fn(DSEL[cand[ii]], Q, w), float)) / len(ii)
        return self.pick(cand, sc, self.quota)

POLICY_REGISTRY = {}
def register_policy(name: str):
    def deco(factory):
        POLICY_REGISTRY[name] = factory
        return factory
    return deco

# ---- Gaussian Process Utility Models ---------------------------------------------
def _sqd(A, B):
    return np.maximum((A * A).sum(1)[:, None] + (B * B).sum(1)[None, :] - 2 * A @ B.T, 0.0)

def gp_fit_predict(Xo, yo, Xall, ls_mult=(0.5, 1.0, 2.0), noise=(0.05, 0.2, 0.6)):
    n = len(Xo); ymu, ysd = yo.mean(), max(yo.std(), 1e-9); z = (yo - ymu) / ysd
    D = _sqd(Xo, Xo); med = math.sqrt(max(np.median(D[np.triu_indices(n, 1)]), 1e-9))
    best = None
    for lm in ls_mult:
        ls = med * lm; K = np.exp(-0.5 * D / ls ** 2)
        for nz in noise:
            try:
                L = np.linalg.cholesky(K + nz * np.eye(n))
            except np.linalg.LinAlgError:
                continue
            a = solve_triangular(L.T, solve_triangular(L, z, lower=True), lower=False)
            lml = -0.5 * z @ a - np.log(np.diag(L)).sum()
            if best is None or lml > best[0]:
                best = (lml, ls, L, a)
    _, ls, L, a = best
    Ks = np.exp(-0.5 * _sqd(Xall, Xo) / ls ** 2)
    mean = Ks @ a; V = solve_triangular(L, Ks.T, lower=True)
    var = np.maximum(1.0 - (V ** 2).sum(0), 1e-9)
    return mean * ysd + ymu, np.sqrt(var) * ysd

class GPUtility(Policy):
    name = "GP-TS (ours)"
    def __init__(self, ep, cap, fb, seed, mode="ts", quota=False, density=True, beta=1.0, pca_dim=8, m_max=800):
        super().__init__(ep, cap, fb, seed)
        self.mode, self.quota, self.density, self.beta, self.pca_dim, self.m_max = mode, quota, density, beta, pca_dim, m_max
    def features(self, cand):
        E = DSEL[cand]; Ec = E - E.mean(0)
        w, V = np.linalg.eigh(Ec.T @ Ec); Z = Ec @ V[:, -self.pca_dim:]
        cols = [Z / (Z.std(0) + 1e-9)]
        if self.density:
            dn = knn_density(E, k=5)
            cols.append(((dn - dn.mean()) / (dn.std() + 1e-9))[:, None])
        return np.hstack(cols)
    def select(self, cand, t):
        obs = np.where(self.exposed[cand])[0]
        if len(obs) < 30: return self.pick(cand, self.rng.rand(len(cand)), self.quota)
        y_all = self.rate(cand[obs])
        if y_all.std() < 1e-9: return self.pick(cand, self.rng.rand(len(cand)), self.quota)
        if len(obs) > self.m_max:
            keep = self.rng.choice(len(obs), self.m_max, replace=False); obs, y_all = obs[keep], y_all[keep]
        X = self.features(cand)
        mean, std = gp_fit_predict(X[obs], y_all, X)
        if self.mode == "ts": s = mean + std * self.rng.randn(len(cand))
        elif self.mode == "ucb": s = mean + self.beta * std
        else: s = mean
        return self.pick(cand, s, self.quota)

class GPWithinDomainQuota(GPUtility):
    name = "GP-WithinDom+Quota"
    def __init__(self, ep, cap, fb, seed):
        super().__init__(ep, cap, fb, seed, mode="mean", quota=True, density=True, pca_dim=24, m_max=600)
    def select(self, cand, t):
        sc = np.zeros(len(cand)); doms = self.ep.doc_dom[cand]
        for d in np.unique(doms):
            ii = np.where(doms == d)[0]; cd = cand[ii]
            obs = np.where(self.exposed[cd])[0]
            dens = knn_density(DSEL[cd], k=5)
            if len(obs) >= 20 and self.rate(cd[obs]).std() > 1e-9:
                X = self.features(cd)
                mu, _ = gp_fit_predict(X[obs], self.rate(cd[obs]), X)
                sc[ii] = 0.5 * (rankdata(mu) / len(ii)) + 0.5 * (rankdata(dens) / len(ii))
            else:
                sc[ii] = rankdata(dens) / len(ii)
        return self.pick(cand, sc, quota=True)

# ---- GraphPPR & Submodular Facility Location Ablations ---------------------------
def ppr_centrality_score(E, Q=None, w=None, k=10, alpha=0.85, iters=15):
    n = len(E); ke = min(k, n - 1)
    if ke < 1: return np.zeros(n)
    S = E @ E.T; np.fill_diagonal(S, -np.inf)
    nn = np.argpartition(-S, ke, axis=1)[:, :ke]
    rows = np.repeat(np.arange(n), ke); cols = nn.ravel()
    A = np.zeros((n, n), dtype=np.float32); A[rows, cols] = np.maximum(S[rows, cols], 0.0)
    A = np.maximum(A, A.T)
    P = A / np.maximum(A.sum(axis=1, keepdims=True), 1e-9)
    v = np.ones(n, dtype=np.float32) / n
    if Q is not None and len(Q) > 0:
        qs = np.maximum(E @ Q.T, 0.0) @ (w / max(w.sum(), 1e-9))
        if qs.max() > 0: v = 0.5 * v + 0.5 * (qs / qs.sum())
    pi = v.copy()
    for _ in range(iters):
        pi = alpha * (P.T @ pi) + (1.0 - alpha) * v
    return pi

def facility_location_greedy(E: np.ndarray, budget: int, prior_scores: Optional[np.ndarray] = None, lam: float = 0.0) -> np.ndarray:
    N = len(E)
    if budget >= N: return np.arange(N)
    S = np.maximum(E @ E.T, 0.0)
    if prior_scores is not None and prior_scores.max() > prior_scores.min():
        p_norm = (prior_scores - prior_scores.min()) / (prior_scores.max() - prior_scores.min() + 1e-9)
    else:
        p_norm = np.zeros(N)
    cur_max = np.zeros(N, dtype=np.float32); chosen = []; avail = np.ones(N, dtype=bool)
    for _ in range(budget):
        gains = np.maximum(S - cur_max[:, None], 0.0).mean(axis=0) + lam * p_norm
        gains[~avail] = -np.inf
        best = int(np.argmax(gains)); chosen.append(best); avail[best] = False
        cur_max = np.maximum(cur_max, S[:, best])
    return np.array(chosen, dtype=int)

class SubmodFacilityQuota(Policy):
    name = "SubmodFL+Quota"
    def __init__(self, ep, cap, fb, seed, rho=0.20, lam=0.15):
        super().__init__(ep, cap, fb, seed); self.rho, self.lam = rho, lam
    def select(self, cand, t):
        rate = self.rate(cand); k_hot = int(self.rho * self.cap)
        hot = np.where(rate > 0)[0]; hot = hot[np.argsort(-rate[hot])][:k_hot]
        rest = np.setdiff1d(np.arange(len(cand)), hot); R = self.cap - len(hot); rc = cand[rest]
        doms = self.ep.doc_dom[rc]; ud = np.unique(doms); share = R // len(ud); chosen = []
        for d in ud:
            ii = np.where(doms == d)[0]; Ed = DSEL[rc[ii]]
            dp = knn_density(Ed, k=5)
            sel = facility_location_greedy(Ed, min(share, len(ii)), prior_scores=dp, lam=self.lam)
            chosen.append(ii[sel])
        chosen = np.concatenate(chosen); need = R - len(chosen)
        if need > 0:
            left = np.setdiff1d(np.arange(len(rc)), chosen)
            dl = knn_density(DSEL[rc[left]], k=5)
            chosen = np.concatenate([chosen, left[np.argsort(-dl)[:need]]])
        return np.concatenate([cand[hot], rc[chosen]])

# ---- Headline Method: CAGE (Centrality-Anchored Gated Eviction) ------------------
def hub_density_centrality(E: np.ndarray) -> np.ndarray:
    """
    Training-free O(N^2) structural centrality prior:
    0.45 * Rank(kNN-Density_5) + 0.45 * Rank(kNN-InDegree_10) + 0.10 * Rank(Medoid-Sim).
    """
    n = len(E)
    if n <= 2: return np.ones(n, dtype=np.float32)
    S = E @ E.T; np.fill_diagonal(S, -1.0)
    kd = min(5, n - 1)
    dens = np.partition(S, -kd, axis=1)[:, -kd:].mean(axis=1)
    kh = min(10, n - 1)
    nn = np.argpartition(-S, kh, axis=1)[:, :kh]
    hub = np.bincount(nn.ravel(), minlength=n).astype(np.float32)
    centroid = E.mean(axis=0); centroid /= max(np.linalg.norm(centroid), 1e-9)
    cent_sim = E @ centroid
    c_score = 0.45 * (rankdata(dens) / n) + 0.45 * (rankdata(hub) / n) + 0.10 * (rankdata(cent_sim) / n)
    return c_score.astype(np.float32)

class CAGEPolicy(Policy):
    """
    CAGE (Centrality-Anchored Gated Eviction with Domain Quotas):
    1. Multi-Hit Admission Gate (rho = 0.06, hit_thresh = 1.2): Only pins verified multi-hit
       heavy-hitters per domain, ranked by centrality-modulated hit rate.
    2. Structural Core (1 - rho = 0.94): Filled by within-domain Hub-Density-Medoid Centrality.
    """
    name = "CAGE+Quota (ours)"
    def __init__(self, ep, cap: int, fb: str, seed: int, rho: float = 0.06, hit_thresh: float = 1.2):
        super().__init__(ep, cap, fb, seed)
        self.rho = rho
        self.hit_thresh = hit_thresh

    def select(self, cand: np.ndarray, t: int) -> np.ndarray:
        doms = self.ep.doc_dom[cand]
        ud = np.unique(doms)
        n_dom = len(ud)
        share = self.cap // n_dom

        C = np.zeros(len(cand), dtype=np.float32)
        for d in ud:
            ii = np.where(doms == d)[0]
            C[ii] = hub_density_centrality(DSEL[cand[ii]])

        if self.rho <= 0:
            return self.pick(cand, C, quota=True)

        rate = self.rate(cand)
        hits = self.hits[cand]
        chosen = []
        hot_per_dom = max(1, int(self.rho * share))

        for d in ud:
            ii = np.where(doms == d)[0]
            r_d, h_d, c_d = rate[ii], hits[ii], C[ii]
            hot_cand = np.where(h_d >= self.hit_thresh)[0]
            if len(hot_cand) > 0:
                hot_score = r_d[hot_cand] * (0.5 + 0.5 * c_d[hot_cand])
                k_h = min(hot_per_dom, len(hot_cand))
                top_h = hot_cand[np.argsort(-hot_score)[:k_h]]
            else:
                top_h = np.zeros(0, dtype=int)

            rem_d = np.setdiff1d(np.arange(len(ii)), top_h)
            need_d = min(share - len(top_h), len(rem_d))
            top_c = rem_d[np.argsort(-c_d[rem_d])[:need_d]]
            chosen.append(ii[np.concatenate([top_h, top_c])])

        chosen = np.concatenate(chosen)
        need = self.cap - len(chosen)
        if need > 0:
            rest = np.setdiff1d(np.arange(len(cand)), chosen)
            chosen = np.concatenate([chosen, rest[np.argsort(-C[rest])[:need]]])
        return cand[chosen]

# Register v11/v12 policies
register_policy("GraphPPR+Quota")(lambda ep, cap, fb, seed: ScorePolicy(ep, cap, fb, seed, ppr_centrality_score, per_domain=True, quota=True))
register_policy("GP-WithinDom+Quota")(lambda ep, cap, fb, seed: GPWithinDomainQuota(ep, cap, fb, seed))
register_policy("SubmodFL-Pure+Quota")(lambda ep, cap, fb, seed: SubmodFacilityQuota(ep, cap, fb, seed, rho=0.0, lam=0.15))
register_policy("SubmodFL+Quota")(lambda ep, cap, fb, seed: SubmodFacilityQuota(ep, cap, fb, seed, rho=0.20, lam=0.15))
register_policy("HubDensity-Pure+Quota")(lambda ep, cap, fb, seed: CAGEPolicy(ep, cap, fb, seed, rho=0.0))
register_policy("CAGE+Quota (ours)")(lambda ep, cap, fb, seed: CAGEPolicy(ep, cap, fb, seed, rho=0.06, hit_thresh=1.2))

def make_policies(ep: Episode, fb: str, seed: int, cap: int) -> Dict[str, Policy]:
    P = {
        "Unbounded (reference)": Unbounded(ep, N_DOCS, fb, seed),
        "Oracle (uses test labels)": Oracle(ep, cap, fb, seed),
        "FIFO": FIFO(ep, cap, fb, seed),
        "Random": RandomEvict(ep, cap, fb, seed),
        "Reservoir": Reservoir(ep, cap, fb, seed),
        "Quota-Random": QuotaRandom(ep, cap, fb, seed),
        "LRU": LRU(ep, cap, fb, seed),
        "LFU": LFU(ep, cap, fb, seed),
        "DecayedHit": DecayedHit(ep, cap, fb, seed),
        "DecayedHit+Quota": DecayedHitQuota(ep, cap, fb, seed),
        "DecayedHit (recency tie-break)": DecayedHitRecent(ep, cap, fb, seed),
        "K-Center (diversity)": KCenter(ep, cap, fb, seed),
        "K-Center+Quota (stratified coreset)": KCenter(ep, cap, fb, seed, quota=True),
        "Density": Density(ep, cap, fb, seed),
        "Density+Quota": Density(ep, cap, fb, seed, quota=True),
        "HitDensity+Quota": HitDensity(ep, cap, fb, seed),
        "WindowHit (W-TinyLFU-style)": WindowHit(ep, cap, fb, seed),
        "QueryProx": QueryProx(ep, cap, fb, seed),
        "QueryProx+Quota": QueryProx(ep, cap, fb, seed, quota=True),
        "GP-TS (ours)": GPUtility(ep, cap, fb, seed, mode="ts"),
        "GP-UCB": GPUtility(ep, cap, fb, seed, mode="ucb"),
        "GP-Mean": GPUtility(ep, cap, fb, seed, mode="mean"),
        "GP-TS+Quota": GPUtility(ep, cap, fb, seed, mode="ts", quota=True),
        "GP-TS (no density)": GPUtility(ep, cap, fb, seed, mode="ts", density=False),
    }
    for rho in CFG.mix_rhos:
        P[f"Mix(rho={rho})"] = Mix(ep, cap, fb, seed, rho=rho)
    P["Hubness+Quota"] = ScorePolicy(ep, cap, fb, seed, knn_indegree, per_domain=True, quota=True)
    for n_, f_ in POLICY_REGISTRY.items():
        P[n_] = f_(ep, cap, fb, seed)
    for k, p in P.items():
        p.name = k
    return P

REFERENCE = "CAGE+Quota (ours)"
NOT_BASELINES = {"Unbounded (reference)", "Oracle (uses test labels)"}

In [ ]:
def cl_metrics(R: np.ndarray) -> dict:
    T = R.shape[0]
    aa = R[T - 1].mean()
    bwt = np.mean([R[T - 1, j] - R[j, j] for j in range(T - 1)])
    fgt = np.mean([R[j:T - 1, j].max() - R[T - 1, j] for j in range(T - 1)])
    aia = np.mean([R[t, :t + 1].mean() for t in range(T)])
    return dict(final_ndcg=aa, bwt=bwt, forgetting=fgt, aia=aia)

def run_episode(pol: Policy, ep: Episode, cfg: Config, defer: bool = False):
    T, k = ep.T, cfg.top_k
    Rn, Rr, Rp, Rpc = np.zeros((T, T)), np.zeros((T, T)), np.zeros((T, T)), np.zeros((T, T))
    comp = np.zeros((T, T)); online = []; lat = []; peak = 0
    cache = {}; cache_docs = set()
    for t in range(T):
        if defer: pol.stage(t, ep.tasks[t]["docs"])
        else: pol.arrive(t, ep.tasks[t]["docs"])
        peak = max(peak, len(pol.mem))
        recs, qcounts = [], np.zeros(T)
        for j in range(t + 1):
            idx = ep.stream[t][j]
            recs += [ep.tasks[j]["usage"][i] for i in idx]
            qcounts[j] += len(idx)
        t0 = time.perf_counter()
        ret = retrieve(pol.mem, recs, k)
        lat.append((time.perf_counter() - t0) / len(recs))
        for i, r in enumerate(recs):
            if r["q"] in cache: continue
            if cfg.cache_budget is None:
                cache[r["q"]] = ret[i].copy(); continue
            newd = set(ret[i].tolist()) - cache_docs
            if len(cache_docs) + len(newd) <= cfg.cache_budget:
                cache[r["q"]] = ret[i].copy(); cache_docs |= newd
        _, _, su = score_queries(ret, recs, k)
        online.append(su.mean())
        if pol.fb == "relevance":
            hit = np.concatenate([ret[i][np.isin(ret[i], r["rel"])] for i, r in enumerate(recs)])
        else:
            hit = ret.ravel()
        pol.feedback(t, hit.astype(int), qcounts, np.array([r["q"] for r in recs]))
        if defer: pol.consolidate(t)
        comp[t] = np.bincount(ep.doc_dom[pol.mem], minlength=T)
        for j in range(t + 1):
            tr = ep.tasks[j]["test"]
            nd, rc, _ = score_queries(retrieve(pol.mem, tr, k), tr, k)
            Rn[t, j], Rr[t, j] = nd.mean(), rc.mean()
            ur = ep.tasks[j]["usage"]
            live = retrieve(pol.mem, ur, k)
            ndu, _, _ = score_queries(live, ur, k)
            Rp[t, j] = float((ndu * ep.pop[j]).sum())
            cr = np.array([cache[r["q"]] if r["q"] in cache else live[i] for i, r in enumerate(ur)])
            ndc, _, _ = score_queries(cr, ur, k)
            Rpc[t, j] = float((ndc * ep.pop[j]).sum())
    m = cl_metrics(Rn)
    m["final_recall"] = Rr[T - 1].mean()
    m["final_rep"] = Rp[T - 1].mean()
    m["peak_docs"] = peak
    m["final_rep_cached"] = Rpc[T - 1].mean()
    m["cache_entries"] = len(cache)
    m["cache_extra_docs"] = int(len(np.setdiff1d(np.unique(np.concatenate(list(cache.values()))), pol.mem))) if cache else 0
    m["cache_docs_total"] = int(len(np.unique(np.concatenate(list(cache.values()))))) if cache else 0
    m["online_post"] = float(np.mean(online[2:])) if T > 2 else float(np.mean(online))
    m["online_all"] = float(np.mean(online))
    m["arrive_ms_per_doc"] = 1000 * pol.evict_time / pol.n_seen
    m["retr_ms_per_query"] = 1000 * float(np.mean(lat))
    m["mem_MB"] = len(pol.mem) * DEMB.shape[1] * 4 / 1e6
    return m, Rn, comp

def run_suite(cfg: Config, seeds, fbs, only=None, tag="main", verbose=True, defer=False):
    rows, mats = [], {}
    for fb in fbs:
        for seed in seeds:
            ep = Episode(POOLS, cfg, seed)
            pols = make_policies(ep, fb, seed, cfg.cap)
            for name, pol in pols.items():
                if only is not None and name not in only and name != "Unbounded (reference)":
                    continue
                m, Rn, comp = run_episode(pol, ep, cfg, defer=defer)
                rows.append(dict(tag=tag, fb=fb, seed=seed, strategy=name, zipf=cfg.zipf_s, defer=defer, **m))
                mats[(tag, fb, seed, name)] = (Rn, comp)
            if verbose:
                print(f"[{tag}] fb={fb} seed={seed} done")
    df = pd.DataFrame(rows)
    ceil = df[df.strategy == "Unbounded (reference)"].set_index(["fb", "seed"])["final_ndcg"]
    df["retention"] = [r.final_ndcg / ceil[(r.fb, r.seed)] for r in df.itertuples()]
    return df, mats

def boot_ci(x, n=5000, seed=0):
    x = np.asarray(x, float); r = np.random.default_rng(seed)
    b = r.choice(x, size=(n, len(x)), replace=True).mean(1)
    return x.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)

def holm(p):
    p = np.asarray(p); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for i, idx in enumerate(o):
        run = max(run, (m - i) * p[idx])
        adj[idx] = min(1.0, run)
    return adj

# Sanity checks
ep_sanity = Episode(POOLS, CFG, 0)
for t_task in ep_sanity.tasks:
    assert not ({r["q"] for r in t_task["usage"]} & {r["q"] for r in t_task["test"]}), "Leakage detected!"
print("All leak-free split & capacity sanity checks passed!")

In [ ]:
def auc_score(score, y):
    y = y.astype(bool); n1, n0 = y.sum(), (~y).sum()
    if n1 == 0 or n0 == 0: return np.nan
    return (rankdata(score)[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def audit(cfg: Config, seeds: List[int]) -> pd.DataFrame:
    rows = []
    for seed in seeds:
        ep = Episode(POOLS, cfg, seed)
        for t, task in enumerate(ep.tasks):
            docs = task["docs"]; E = DSEL[docs]
            ytest = np.isin(docs, np.unique(np.concatenate([r["rel"] for r in task["test"]])))
            urel = np.isin(docs, np.unique(np.concatenate([r["rel"] for r in task["usage"]])))
            sq = E @ QSEL[[r["q"] for r in task["usage"]]].T
            dens = knn_density(E)
            sig = {
                "kNN density (typical docs)": dens,
                "outlierness": -dens,
                "max-sim to usage queries": sq.max(1),
                "top-3 sim to usage queries": np.sort(sq, 1)[:, -3:].mean(1),
                "relevant to a usage query": urel.astype(float),
            }
            for k_, v in sig.items():
                rows.append(dict(seed=seed, domain=ep.names[t], signal=k_, value=auc_score(v, ytest)))
            m = ~urel
            for k_ in ["kNN density (typical docs)", "max-sim to usage queries"]:
                rows.append(dict(seed=seed, domain=ep.names[t], signal=k_ + " | docs not usage-relevant", value=auc_score(sig[k_][m], ytest[m])))
            rows.append(dict(seed=seed, domain=ep.names[t], signal="REPEAT RATE (test-relevant docs also usage-relevant)", value=(ytest & urel).sum() / max(1, ytest.sum())))
            rows.append(dict(seed=seed, domain=ep.names[t], signal="BASE RATE (docs relevant to a test query)", value=ytest.mean()))
    return pd.DataFrame(rows)

AUD = audit(CFG, CFG.seeds)
AUD.to_csv(os.path.join(CFG.out_dir, "audit.csv"), index=False)
tab = AUD.pivot_table(index="signal", columns="domain", values="value", aggfunc="mean")
tab["ALL"] = AUD.groupby("signal")["value"].mean()
print("=== TABLE 1: PRE-EVICTION PREDICTABILITY AUDIT (ROC-AUC FOR NOVEL QUERIES) ===")
print(tab.round(3).to_string())

In [ ]:
t0 = time.time()
DF, MATS = run_suite(CFG, CFG.seeds, CFG.feedback_modes, tag="main")
print(f"Main exploratory suite finished in {time.time() - t0:.1f}s")
DF.to_csv(os.path.join(CFG.out_dir, "main_results.csv"), index=False)

for fb in CFG.feedback_modes:
    rows = []
    for s, g in DF[DF.fb == fb].groupby("strategy"):
        m, lo, hi = boot_ci(g["final_ndcg"].values)
        rows.append(dict(
            strategy=s, novel_ndcg=m, ci_lo=lo, ci_hi=hi,
            rep_ndcg=g.final_rep.mean(), rep_cached=g.final_rep_cached.mean(),
            bwt=g.bwt.mean(), forgetting=g.forgetting.mean(),
            online_post=g.online_post.mean(), ms_per_doc=g.arrive_ms_per_doc.mean()
        ))
    print(f"\n=== TABLE 2: EXPLORATORY RESULTS (FEEDBACK = {fb}) ===")
    print(pd.DataFrame(rows).sort_values("novel_ndcg", ascending=False).round(4).to_string(index=False))

In [ ]:
CAGE_SET = {
    "Random", "FIFO", "DecayedHit", "K-Center+Quota (stratified coreset)",
    "Density+Quota", "Hubness+Quota", "HitDensity+Quota",
    "HubDensity-Pure+Quota", "CAGE+Quota (ours)", "Oracle (uses test labels)"
}

def _pair_stats(pv, a, b):
    d = (pv[a] - pv[b]).values
    p = wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    m, lo, hi = boot_ci(d)
    return m, lo, hi, p

def print_verdicts(df: pd.DataFrame, title: str):
    pn = df.pivot(index="seed", columns="strategy", values="final_ndcg")
    pr = df.pivot(index="seed", columns="strategy", values="final_rep")
    ha = _pair_stats(pn, "CAGE+Quota (ours)", "Random")
    hb = _pair_stats(pr, "CAGE+Quota (ours)", "Density+Quota")
    hc = _pair_stats(pn, "CAGE+Quota (ours)", "Density+Quota")
    hd = _pair_stats(pn, "HubDensity-Pure+Quota", "Random")
    hk = _pair_stats(pn, "Random", "K-Center+Quota (stratified coreset)") if "K-Center+Quota (stratified coreset)" in pn.columns else None
    ph = holm(np.array([ha[3], hb[3], hd[3]]))
    print("\n" + "=" * 85)
    print(title)
    print("=" * 85)
    for nm, h, pa in [
        ("H_a CAGE+Quota (ours) vs Random (novel nDCG)", ha, ph[0]),
        ("H_b CAGE+Quota (ours) vs Density+Quota (repeated nDCG)", hb, ph[1]),
        ("H_d HubDensity-Pure+Quota vs Random (novel nDCG)", hd, ph[2]),
    ]:
        status = "CONFIRMED" if (pa < 0.05 and h[1] > 0) else "NOT confirmed"
        print(f"{nm}: diff = {h[0]:+.4f} [95% CI {h[1]:+.4f}, {h[2]:+.4f}], raw p = {h[3]:.4f}, Holm p = {pa:.4f} -> {status}")
    status_c = "NON-INFERIOR (margin 0.01)" if hc[1] > -0.01 else "non-inferiority NOT shown"
    print(f"H_c CAGE+Quota (ours) vs Density+Quota (novel non-inferiority): diff = {hc[0]:+.4f} [95% CI {hc[1]:+.4f}, {hc[2]:+.4f}] -> {status_c}")
    if hk is not None:
        print(f"H_k Random vs K-Center+Quota (coreset outlier penalty): diff = {hk[0]:+.4f} [95% CI {hk[1]:+.4f}, {hk[2]:+.4f}], raw p = {hk[3]:.4f}")
    s = df.groupby("strategy")[["final_ndcg", "final_rep", "final_rep_cached", "bwt", "forgetting", "arrive_ms_per_doc"]].mean()
    s["mixed_0.25"] = 0.25 * s["final_rep"] + 0.75 * s["final_ndcg"]
    s["mixed_0.50"] = 0.50 * s["final_rep"] + 0.50 * s["final_ndcg"]
    print(s.round(4).sort_values("mixed_0.25", ascending=False).to_string())

if MODE == "full":
    # Load all 8 domains once (first 5 = Base-5, all 8 = Extended-8)
    EMB_RET = make_embedder("BAAI/bge-small-en-v1.5")
    EMB_E5 = make_embedder("intfloat/e5-small-v2")
    POOLS_8_ALL = [build_pool_real(d, EMB_RET, EMB_E5) for d in DOMAIN_SETS["extended8"]]

    # 1. Regime 1: Base-5 Same-Embedder (BGE -> BGE, Seeds 600-614)
    POOLS = POOLS_8_ALL[:5]
    DOC_OFF = np.cumsum([0] + [p.n_docs for p in POOLS])
    Q_OFF = np.cumsum([0] + [p.Q.shape[0] for p in POOLS])
    DEMB = np.vstack([p.D for p in POOLS]); QEMB = np.vstack([p.Q for p in POOLS])
    DSEL, QSEL = DEMB, QEMB; N_DOCS = DEMB.shape[0]
    CFG_B5_BGE = replace(CFG, domains=list(DOMAIN_SETS["base5"]), sel_model=None, cap=2000)
    DF_B5_BGE, _ = run_suite(CFG_B5_BGE, list(range(600, 615)), ["relevance"], only=CAGE_SET, tag="b5_bge_15", verbose=False)
    print_verdicts(DF_B5_BGE, "1. BASE-5 SAME-EMBEDDER (BGE->BGE, 15 CONFIRMATORY SEEDS 600-614)")

    # 2. Regime 2: Base-5 Cross-Embedder Control (E5 -> BGE, Seeds 600-614)
    DSEL = np.vstack([p.Ds for p in POOLS]); QSEL = np.vstack([p.Qs for p in POOLS])
    CFG_B5_E5 = replace(CFG_B5_BGE, sel_model="intfloat/e5-small-v2")
    DF_B5_E5, _ = run_suite(CFG_B5_E5, list(range(600, 615)), ["relevance"], only=CAGE_SET, tag="b5_e5_15", verbose=False)
    print_verdicts(DF_B5_E5, "2. BASE-5 CROSS-EMBEDDER CONTROL (E5->BGE, 15 CONFIRMATORY SEEDS 600-614)")

    # 3. Regime 3: Extended-8 Continual Stream (E5 -> BGE, Seeds 700-719)
    POOLS = POOLS_8_ALL
    DOC_OFF = np.cumsum([0] + [p.n_docs for p in POOLS])
    Q_OFF = np.cumsum([0] + [p.Q.shape[0] for p in POOLS])
    DEMB = np.vstack([p.D for p in POOLS]); QEMB = np.vstack([p.Q for p in POOLS])
    DSEL = np.vstack([p.Ds for p in POOLS]); QSEL = np.vstack([p.Qs for p in POOLS])
    N_DOCS = DEMB.shape[0]
    CFG_E8 = replace(CFG, domains=list(DOMAIN_SETS["extended8"]), sel_model="intfloat/e5-small-v2", cap=3200)
    DF_E8_20, _ = run_suite(CFG_E8, list(range(700, 720)), ["relevance"], only=CAGE_SET, tag="ext8_20", verbose=False)
    print_verdicts(DF_E8_20, "3. EXTENDED-8 CONTINUAL STREAM (8 DOMAINS, 8000 DOCS, 20 CONFIRMATORY SEEDS 700-719)")
else:
    DF_SMOKE, _ = run_suite(CFG, [600, 601, 602, 603, 604, 605], ["relevance"], only=CAGE_SET, tag="smoke_confirm", verbose=False)
    print_verdicts(DF_SMOKE, "SMOKE MODE CONFIRMATORY CHECK (6 SEEDS)")